# 00 - EDA y Calidad de Datos\n\nExploración inicial del archivo `venta_tiendas.csv` y revisión de calidad para definir reglas de transformación en Silver.

# Configuración base

Este notebook está preparado para ejecutarse en Google Colab conectado directamente a **Google Cloud Storage (GCS)** mediante autenticación nativa, analizando los datos crudos antes de su procesamiento.

Estructura esperada en el Bucket (`gs://data-lake-retail/`):

```text
data-lake-retail/
├── raw/
│   ├── venta_tiendas.csv         <-- (Origen: Archivo principal analizado en este EDA)
│   ├── Maestro_Producto.csv      <-- (Origen: Archivos maestros disponibles)
│   ├── Maestro_Tienda.csv
│   └── venta_ecom.csv
├── bronze/
├── silver/
├── gold/
└── evidencias/

In [1]:
# 1. Autenticación con Google Cloud
from google.colab import auth
auth.authenticate_user()
print("Autenticación con GCP exitosa.")

# 2. Instalación de dependencias limpias
!pip uninstall -y dataproc-spark-connect opentelemetry-api importlib-metadata pyspark delta-spark > /dev/null
!pip install -q importlib-metadata==8.0.0 pyspark==3.4.1 delta-spark==2.4.0

# 3. 🔥 SOLUCIÓN: Descargar el conector GCS manualmente a la carpeta de PySpark
import pyspark
import os
pyspark_jars_dir = os.path.join(pyspark.__path__[0], "jars")
!wget -q https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar -P {pyspark_jars_dir}
print("Conector GCS descargado correctamente.")

# 4. Configuración de Spark con soporte Delta y GCS
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("Forus_Fase2_BigData")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    # Indicamos a Spark cómo procesar las rutas gs://
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    # Colab usa la autenticación por defecto de la máquina virtual
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

# 5. Definición de Rutas en la Nube
NOMBRE_BUCKET = "data-lake-retail"


RUTA_BASE = f"gs://{NOMBRE_BUCKET}" 

RUTA_RAW = f"{RUTA_BASE}/raw"
RUTA_BRONZE = f"{RUTA_BASE}/bronze"
RUTA_SILVER = f"{RUTA_BASE}/silver"
RUTA_GOLD = f"{RUTA_BASE}/gold"
RUTA_EVIDENCIAS = f"{RUTA_BASE}/evidencias"

print("Ruta base configurada:", RUTA_BASE)

# 6. Lectura del archivo CSV
# Esto construirá exactamente: gs://data-lake-retail/raw/venta_tiendas.csv
ruta_csv = f"{RUTA_RAW}/venta_tiendas.csv" 
print(f"Leyendo datos desde: {ruta_csv}")

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(ruta_csv)
)
# 7. Muestra de resultados
print("Registros:", df.count())
print("Columnas:", len(df.columns))
df.printSchema()
df.show(5, truncate=False)

Autenticación con GCP exitosa.
Conector GCS descargado correctamente.
Spark version: 3.4.1
Ruta base configurada: gs://data-lake-retail
Leyendo datos desde: gs://data-lake-retail/raw/venta_tiendas.csv
Registros: 2250970
Columnas: 11
root
 |-- id_canal: integer (nullable = true)
 |-- numero_transaccion: integer (nullable = true)
 |-- numero_pos: integer (nullable = true)
 |-- numero_boleta: integer (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: integer (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: integer (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- venta: integer (nullable = true)
 |-- costo: integer (nullable = true)

+--------+------------------+----------+-------------+-------------------------+----------------------+--------------+-----------+--------+-----+-----+
|id_canal|numero_transaccion|numero_pos|numero_boleta|fecha_transaccion        |cod_tienda_facturacion|tipo_documento

## 2. Revisión de valores nulos

In [2]:
from pyspark.sql.functions import col, sum as spark_sum, when

nulos = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

nulos.show(truncate=False)


+--------+------------------+----------+-------------+-----------------+----------------------+--------------+-----------+--------+-----+-----+
|id_canal|numero_transaccion|numero_pos|numero_boleta|fecha_transaccion|cod_tienda_facturacion|tipo_documento|id_producto|unidades|venta|costo|
+--------+------------------+----------+-------------+-----------------+----------------------+--------------+-----------+--------+-----+-----+
|0       |0                 |0         |0            |0                |0                     |0             |0          |0       |0    |0    |
+--------+------------------+----------+-------------+-----------------+----------------------+--------------+-----------+--------+-----+-----+



## 3. Revisión de duplicados por transacción y producto

In [3]:
claves = ["id_canal", "numero_transaccion", "numero_pos", "numero_boleta", "id_producto"]

total = df.count()
distintos = df.select(claves).distinct().count()

print("Total registros:", total)
print("Combinaciones únicas por clave:", distintos)
print("Posibles duplicados:", total - distintos)


Total registros: 2250970
Combinaciones únicas por clave: 2246621
Posibles duplicados: 4349


## 4. Estadísticas descriptivas de variables numéricas

In [4]:
columnas_numericas = ["unidades", "venta", "costo"]

df.select(columnas_numericas).describe().show()


+-------+------------------+-----------------+------------------+
|summary|          unidades|            venta|             costo|
+-------+------------------+-----------------+------------------+
|  count|           2250970|          2250970|           2250970|
|   mean|0.8490846168540673|23822.78605667779| 9772.421610239142|
| stddev|0.6309010987041614|36437.25825146752|13480.440706964917|
|    min|               -45|         -2646681|           -842708|
|    max|                85|         16070786|           4894690|
+-------+------------------+-----------------+------------------+



## 5. Revisión del formato de fecha\n\nEn el archivo original la fecha viene en la columna `fecha_transaccion`, por ejemplo `14/03/2016 12:00:00 AM CL`. En Bronze se conservará igual y en Silver se transformará a `fecha_venta` estándar.

In [5]:
df.select("fecha_transaccion").distinct().show(10, truncate=False)


+-------------------------+
|fecha_transaccion        |
+-------------------------+
|07/07/2023 12:00:00 AM CL|
|20/06/2020 12:00:00 AM CL|
|17/11/2017 12:00:00 AM CL|
|26/03/2022 12:00:00 AM CL|
|30/01/2016 12:00:00 AM CL|
|25/11/2015 12:00:00 AM CL|
|17/05/2018 12:00:00 AM CL|
|15/04/2015 12:00:00 AM CL|
|08/01/2026 12:00:00 AM CL|
|04/07/2019 12:00:00 AM CL|
+-------------------------+
only showing top 10 rows



## 6. Métricas base para evidencia

In [6]:
from pyspark.sql.functions import countDistinct, min as spark_min, max as spark_max

metricas = df.agg(
    countDistinct("numero_boleta").alias("boletas_distintas"),
    countDistinct("id_producto").alias("productos_distintos"),
    countDistinct("cod_tienda_facturacion").alias("tiendas_distintas"),
    spark_min("fecha_transaccion").alias("fecha_min_raw"),
    spark_max("fecha_transaccion").alias("fecha_max_raw")
)

metricas.show(truncate=False)


+-----------------+-------------------+-----------------+-------------------------+-------------------------+
|boletas_distintas|productos_distintos|tiendas_distintas|fecha_min_raw            |fecha_max_raw            |
+-----------------+-------------------+-----------------+-------------------------+-------------------------+
|865065           |420699             |469              |01/02/2023 12:00:00 AM CL|31/10/2025 12:00:00 AM CL|
+-----------------+-------------------+-----------------+-------------------------+-------------------------+

